<div style="background-color: #FFFF00; color: #000000; padding: 12px 18px; font-family: Arial, Helvetica, sans-serif;">
<strong style="color: #000000;">MINE-4101 Ciencia de Datos Aplicada · Universidad de los Andes · 2026-20</strong><br>
<span style="color: #000000;">Proyecto final · Entrega final · Actividad 10</span>
</div>

# Construcción y evaluación del modelo

Modelos que estiman la probabilidad de que cada predio pague el impuesto predial, y su evaluación frente a la línea base.

**Integrantes**

- Daniel Felipe Triviño Santana
- Nicolas Bedoya Figueroa
- Joubert Daniel Álvarez
- Gabriel Bejarano

**Qué pide el enunciado** (actividad 10, 20 % de la entrega final): entrenar modelos con al menos tres algoritmos y distintos conjuntos de hiperparámetros, evaluarlos cuantitativamente con las métricas del enfoque analítico y, en la medida de lo posible, cualitativamente, y establecer oportunidades de mejora.

**Punto de partida:** el enfoque analítico (sección 4 del [reporte](../reporte/reporte.pdf)) definió tres algoritmos (regresión logística, un árbol de decisión poco profundo y *gradient boosting*), la línea base (repetir el pago de 2025) y las métricas: ROC-AUC, log-loss y curva de calibración, reportadas por grupo (pagó en 2025, no pagó, predio nuevo). Usa los datos del [notebook 08](08-preparacion-de-datos.ipynb) y la partición del [notebook 09](09-estrategia-de-validacion.ipynb). El mejor modelo se guarda para el [producto de datos](../producto/README.md).

**Cómo hacerlo:** el [sitio de prácticas del curso](https://datascience-uniandes.github.io/PracticasMachineLearning/) es la referencia principal; para esta actividad, [práctica 2: Clasificación](https://datascience-uniandes.github.io/PracticasMachineLearning/talleres/practica-2/), [práctica 5: Interpretabilidad](https://datascience-uniandes.github.io/PracticasMachineLearning/talleres/practica-5/) y [práctica 7: Justicia](https://datascience-uniandes.github.io/PracticasMachineLearning/talleres/practica-7/), y las guías [regresión logística](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/regresion-logistica/), [árbol de decisión](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/arbol-decision/), [gradient boosting](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/gradient-boosting/), [curva ROC](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/curva-roc/), [comparar métricas](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/comparar-metricas/), [impacto en el negocio](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/impacto-negocio/), [SHAP](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/shap/) y [guardar el modelo](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/guardar-modelo/). Si el notebook se aparta de ellas, lo dice y explica por qué.

**Estado:** andamiaje. Las etapas tienen su propósito; el código y la interpretación están por escribir.

## Tabla de contenido

| | Etapa | Qué se hace |
| --- | --- | --- |
| 1 | [Librerías y configuración](#librerias) | Librerías, rutas y estilo |
| 2 | [Carga de los datos y la partición](#carga) | Leer las salidas de los notebooks 08 y 09 |
| 3 | [Línea base](#base) | Repetir el pago de 2025 |
| 4 | [Algoritmos e hiperparámetros](#modelos) | Tres algoritmos, varios conjuntos de hiperparámetros, comparados en validación |
| 5 | [Evaluación cuantitativa](#cuantitativa) | El modelo elegido en prueba: ROC-AUC, log-loss y calibración, por grupo |
| 6 | [Evaluación cualitativa](#cualitativa) | Qué variables pesan y en qué casos se equivoca |
| 7 | [Del modelo al recaudo esperado](#recaudo) | La suma del impuesto por la probabilidad de pago, frente a lo recaudado |
| 8 | [Oportunidades de mejora](#mejora) | Qué haría mejor al modelo |
| 9 | [Guardar el mejor modelo](#guardar) | El archivo que usa el producto |

<a id="librerias"></a>

## 1. Librerías y configuración

Las librerías están en el entorno del proyecto (ver el [README](../../README.md#entorno-de-python)). Las rutas de los archivos que pasan de un notebook a otro, la semilla y las columnas reservadas están en [utilidades_modelado.py](utilidades_modelado.py), junto a este notebook.

In [ ]:
import platform

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# Rutas, semilla, columnas reservadas y estilo (utilidades_modelado.py, en esta carpeta)
import utilidades_modelado as um

um.configurar()

In [ ]:
# Versiones con las que se ejecutó el notebook
print(f"Python {platform.python_version()}")
for libreria in (np, pd, mpl, sns, sklearn):
    print(f"{libreria.__name__} {libreria.__version__}")

<a id="carga"></a>

## 2. Carga de los datos y la partición

**Salidas.** El repositorio es público: ninguna salida puede mostrar datos personales ni nada que identifique al municipio, y los valores de `um.COLUMNAS_RESERVADAS` no se muestran (ver la etapa 2 del [notebook 06](../../entrega-1/notebooks/06-entendimiento-de-los-datos.ipynb)). La variable objetivo y las que salen de la facturación se muestran solo agregadas.

In [ ]:
um.exigir(um.ARCHIVO_PREPARADO, "ejecute antes el notebook 08-preparacion-de-datos.ipynb")
um.exigir(um.ARCHIVO_PARTICION, "ejecute antes el notebook 09-estrategia-de-validacion.ipynb")
preparados = pd.read_parquet(um.ARCHIVO_PREPARADO)
particion = pd.read_parquet(um.ARCHIVO_PARTICION)
preparados.shape, particion.shape

<a id="base"></a>

## 3. Línea base

Predecir para 2026 el pago de 2025. Todas las métricas se comparan con ella.

In [ ]:
# Pendiente: métricas de la línea base en validación y prueba, por grupo

<a id="modelos"></a>

## 4. Algoritmos e hiperparámetros

Cada algoritmo dentro de un `Pipeline` con las transformaciones de la etapa 6 del notebook 08, ajustadas solo con entrenamiento. Una tabla con cada combinación de hiperparámetros y sus métricas en validación.

In [ ]:
# Pendiente: regresión logística (por ejemplo, regularización C)

In [ ]:
# Pendiente: árbol de decisión (por ejemplo, max_depth y min_samples_leaf)

In [ ]:
# Pendiente: gradient boosting (por ejemplo, learning_rate, max_depth y max_iter)

In [ ]:
# Pendiente: tabla comparativa en validación y elección del mejor modelo

<a id="cuantitativa"></a>

## 5. Evaluación cuantitativa

El modelo elegido, evaluado una sola vez en prueba, frente a la línea base y por grupo.

In [ ]:
# Pendiente: ROC-AUC, log-loss y curva de calibración en prueba

<a id="cualitativa"></a>

## 6. Evaluación cualitativa

Importancia de las variables (por permutación o coeficientes), ejemplos agregados de aciertos y errores, y si el comportamiento tiene sentido para el negocio.

In [ ]:
# Pendiente: interpretación del modelo

<a id="recaudo"></a>

## 7. Del modelo al recaudo esperado

$R_{\text{esperado}} = \sum_i \mathit{IPU}_i\,p_i$: con el impuesto facturado en 2026 y las probabilidades del modelo en prueba, comparado con lo efectivamente pagado. Solo cifras agregadas.

In [ ]:
# Pendiente: recaudo esperado frente al observado, en prueba

<a id="mejora"></a>

## 8. Oportunidades de mejora

Pendiente: qué datos, variables o técnicas mejorarían el modelo.

<a id="guardar"></a>

## 9. Guardar el mejor modelo

El pipeline completo (transformaciones y modelo) en `entrega-2/producto/artefactos/modelo.joblib`, que no se sube al repositorio.

In [ ]:
# Pendiente: um.CARPETA_ARTEFACTOS.mkdir(exist_ok=True); joblib.dump(mejor, um.ARCHIVO_MODELO)